# Bank Customer Churn Analysis
### From Raw Data to Interactive Dashboard

| | |
|:---|:---|
| **Program** | NPower Canada — Data Analytics Program, Toronto |
| **Dataset** | Bank Customer Churn · 10,000 records · 13 features |
| **Tools** | Python · Pandas · NumPy · Plotly · Dash · Scikit-learn |
| **Goal** | Identify key drivers of customer churn, deliver insights through an interactive dashboard, and predict churn probability with ML models |

---

## Table of Contents
1. [Setup & Libraries](#1-setup--libraries)
2. [Load Raw Data](#2-load-raw-data)
3. [Data Cleaning & Wrangling](#3-data-cleaning--wrangling)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Interactive Dashboard](#5-interactive-dashboard)
6. [Key Findings — EDA](#6-key-findings--recommendations)
7. [Predictive Modelling](#7-predictive-modelling--churn-probability-scoring)
8. [Combined Findings & Next Steps](#8-combined-findings--next-steps)

---
## 1. Setup & Libraries

Import all required libraries. **Pandas** and **NumPy** handle data manipulation; **Plotly** and **Dash** power the interactive visualizations and dashboard.

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully")

Libraries loaded successfully


---
## 2. Load Raw Data

We start with **two messy source files** that simulate a real-world scenario where data arrives from different systems:

| File | Description | Known Issues |
|:-----|:------------|:-------------|
| `Bank_Churn_Messy.xlsx` | Demographics & account info (8 cols) | Missing values · `€` symbol · inconsistent geography · 1 duplicate |
| `Bank_Churn_Messy_Customer_Info.csv` | Full customer records (13 cols) | Text `Yes/No` in binary column · currency encoding · empty column · 1 duplicate |

In [3]:
df_xlsx = pd.read_excel("data/Bank_Churn_Messy.xlsx")
df_csv  = pd.read_csv("data/Bank_Churn_Messy_Customer_Info.csv", encoding="latin1")

print(f"XLSX shape: {df_xlsx.shape} columns: {df_xlsx.columns.tolist()}")
print(f"CSV shape: {df_csv.shape} columns: {df_csv.columns.tolist()}")

XLSX shape: (10001, 8) columns: ['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'EstimatedSalary']
CSV shape: (10001, 14) columns: ['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'EstimatedSalary', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited', 'Unnamed: 13']


### 2.1 Preview Raw Files

In [4]:
print("Bank_Churn_Messy.xlsx — first 5 rows")
df_xlsx.head()

Bank_Churn_Messy.xlsx — first 5 rows


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
0,15634602,Hargrave,619,FRA,Female,42.00,2,€101348.88
1,15647311,Hill,608,Spain,Female,41.00,1,€112542.58
2,15619304,Onio,502,French,Female,42.00,8,€113931.57
3,15701354,Boni,699,FRA,Female,39.00,1,€93826.63
4,15737888,Mitchell,850,Spain,Female,43.00,2,€79084.1


In [5]:
print("Bank_Churn_Messy_Customer_Info.csv — first 5 rows")
df_csv.head()

Bank_Churn_Messy_Customer_Info.csv — first 5 rows


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Unnamed: 13
0,15634602,Hargrave,619,FRA,Female,42.00,2,101348.88,0.0,1,Yes,Yes,1,NaN
1,15647311,Hill,608,Spain,Female,41.00,1,112542.58,83807.86,1,Yes,Yes,0,NaN
2,15619304,Onio,502,French,Female,42.00,8,113931.57,159660.8,3,No,No,1,NaN
3,15701354,Boni,699,FRA,Female,39.00,1,93826.63,0.0,2,No,No,0,NaN
4,15737888,Mitchell,850,Spain,Female,43.00,2,79084.1,125510.82,1,Yes,Yes,0,NaN


### 2.2 Data Quality Audit

Before cleaning, we document every quality issue found in both files.

In [6]:
print("━" * 50)
print("  DATA QUALITY AUDIT")
print("━" * 50)

print("\n── XLSX Issues ──")
xlsx_missing = df_xlsx.isnull().sum()
print(f"  Missing values:\n{xlsx_missing[xlsx_missing > 0].to_string()}")
print(f"  Duplicate CustomerIds : {df_xlsx['CustomerId'].duplicated().sum()}")
print(f"  EstimatedSalary sample: {df_xlsx['EstimatedSalary'].head(3).tolist()}  ← currency symbol")
print(f"  Geography unique      : {sorted(df_xlsx['Geography'].unique())}  ← inconsistent")

print("\n── CSV Issues ──")
csv_missing = df_csv.isnull().sum()
print(f"  Missing values:\n{csv_missing[csv_missing > 0].to_string()}")
print(f"  Duplicate CustomerIds : {df_csv['CustomerId'].duplicated().sum()}")
print(f"  IsActiveMember values : {df_csv['IsActiveMember'].unique()}  ← should be 0/1")
print(f"  EstimatedSalary dtype : {df_csv['EstimatedSalary'].dtype}  ← should be float")
empty_cols = df_csv.columns[df_csv.isnull().all()].tolist()
print(f"  Fully empty columns   : {empty_cols}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DATA QUALITY AUDIT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

── XLSX Issues ──
  Missing values:
Surname    3
Age        3
  Duplicate CustomerIds : 1
  EstimatedSalary sample: ['€101348.88', '€112542.58', '€113931.57']  ← currency symbol
  Geography unique      : ['FRA', 'France', 'French', 'Germany', 'Spain']  ← inconsistent

── CSV Issues ──
  Missing values:
Surname            3
Age                3
Unnamed: 13    10001
  Duplicate CustomerIds : 1
  IsActiveMember values : <StringArray>
['Yes', 'No']
Length: 2, dtype: str  ← should be 0/1
  EstimatedSalary dtype : str  ← should be float
  Fully empty columns   : ['Unnamed: 13']


---
## 3. Data Cleaning & Wrangling

### 3.1 Clean XLSX File

| Issue | Fix |
|:------|:----|
| 3 missing `Surname` | Fill with `'Unknown'` |
| 3 missing `Age` | Fill with column mean |
| `EstimatedSalary` has `€` symbol | Strip `€`, convert to `float` |
| `Geography` has `'FRA'` and `'French'` | Map both → `'France'` |
| 1 duplicate `CustomerId` | Keep last occurrence |

In [7]:
# ── Fix missing values
df_xlsx["Surname"] = df_xlsx["Surname"].fillna("Unknown")
mean_age_xlsx = int(df_xlsx["Age"].mean())
df_xlsx["Age"] = df_xlsx["Age"].fillna(mean_age_xlsx).astype(int)
print(f"  ✔ Missing values filled — Age mean used: {mean_age_xlsx}")

# ── Strip currency symbol from EstimatedSalary
df_xlsx["EstimatedSalary"] = (
    df_xlsx["EstimatedSalary"].str.replace("€", "", regex=False).astype(float)
)
print(f"  ✔ EstimatedSalary dtype: {df_xlsx['EstimatedSalary'].dtype}")

# ── Standardize Geography names
df_xlsx["Geography"] = df_xlsx["Geography"].replace({"FRA": "France", "French": "France"})
print(f"  ✔ Geography unique: {sorted(df_xlsx['Geography'].unique())}")

# ── Remove duplicate CustomerIds
before = len(df_xlsx)
df_xlsx = df_xlsx.drop_duplicates(subset="CustomerId", keep="last").reset_index(drop=True)
print(f"  ✔ Duplicates removed: {before - len(df_xlsx)}  |  Final shape: {df_xlsx.shape}")

  ✔ Missing values filled — Age mean used: 38
  ✔ EstimatedSalary dtype: float64
  ✔ Geography unique: ['France', 'Germany', 'Spain']
  ✔ Duplicates removed: 1  |  Final shape: (10000, 8)


### 3.2 Clean CSV File

| Issue | Fix |
|:------|:----|
| 3 missing `Surname` & `Age` | Same strategy as XLSX |
| `EstimatedSalary` encoded as `\x80` (latin1 `€`) | Strip `\x80`, convert to `float` |
| `Geography` inconsistent | Same mapping as XLSX |
| `IsActiveMember` is `'Yes'`/`'No'` text | Map → `1` / `0` |
| Empty column `Unnamed: 13` | Drop |
| 1 duplicate `CustomerId` | Remove |

In [8]:
# ── Drop empty column
df_csv = df_csv.drop(columns=["Unnamed: 13"])
print(f"  ✔ Empty column dropped")

# ── Fix missing values
df_csv["Surname"] = df_csv["Surname"].fillna("Unknown")
mean_age_csv = int(df_csv["Age"].mean())
df_csv["Age"] = df_csv["Age"].fillna(mean_age_csv).astype(int)
print(f"  ✔ Missing values filled — Age mean used: {mean_age_csv}")

# ── Fix currency encoding (latin1 € = \x80)
df_csv["EstimatedSalary"] = (
    df_csv["EstimatedSalary"].str.replace("\x80", "", regex=False).astype(float)
)
print(f"  ✔ EstimatedSalary dtype: {df_csv['EstimatedSalary'].dtype}")
df_csv["Balance"] = (
    df_csv["Balance"].str.replace("\x80", "", regex=False).astype(float)
)
print(f"  ✔ Balance dtype: {df_csv['Balance'].dtype}")

# ── Standardize Geography
df_csv["Geography"] = df_csv["Geography"].replace({"FRA": "France", "French": "France"})
print(f"  ✔ Geography unique: {sorted(df_csv['Geography'].unique())}")

# ── Encode HasCrCard and IsActiveMember: Yes/No → 1/0
df_csv["HasCrCard"] = df_csv["HasCrCard"].map({"Yes": 1, "No": 0})
print(f"  ✔ HasCrCard encoded: {sorted(df_csv['HasCrCard'].unique())}")
df_csv["IsActiveMember"] = df_csv["IsActiveMember"].map({"Yes": 1, "No": 0})
print(f"  ✔ IsActiveMember encoded: {sorted(df_csv['IsActiveMember'].unique())}")

# ── Remove duplicates
before = len(df_csv)
df_csv = df_csv.drop_duplicates(subset="CustomerId", keep="last").reset_index(drop=True)
print(f"  ✔ Duplicates removed: {before - len(df_csv)}  |  Final shape: {df_csv.shape}")


  ✔ Empty column dropped
  ✔ Missing values filled — Age mean used: 38
  ✔ EstimatedSalary dtype: float64
  ✔ Balance dtype: float64
  ✔ Geography unique: ['France', 'Germany', 'Spain']
  ✔ HasCrCard encoded: [np.int64(0), np.int64(1)]
  ✔ IsActiveMember encoded: [np.int64(0), np.int64(1)]
  ✔ Duplicates removed: 1  |  Final shape: (10000, 13)


### 3.3 Feature Engineering

Creating new columns to enable richer segmentation analysis:

| New Column | Logic | Purpose |
|:-----------|:------|:--------|
| `AgeGroup` | Bins: 18–30, 31–40, 41–50, 51–60, 60+ | Age-based churn analysis |
| `BalanceTier` | Bins: Zero, Low, Mid, High | Balance-segment analysis |
| `TenureGroup` | Bins: New (0–2), Mid (3–5), Loyal (6+) | Tenure-based analysis |

In [9]:
df = df_csv.copy()

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Balance"] = pd.to_numeric(df["Balance"], errors="coerce")
df["Tenure"] = pd.to_numeric(df["Tenure"], errors="coerce")

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[17, 30, 40, 50, 60, 100],
    labels=["18–30", "31–40", "41–50", "51–60", "60+"]
)

df["BalanceTier"] = pd.cut(
    df["Balance"],
    bins=[-1, 0, 50_000, 125_000, 300_000],
    labels=["Zero", "Low (<50k)", "Mid (50–125k)", "High (>125k)"]
)

df["TenureGroup"] = pd.cut(
    df["Tenure"],
    bins=[-1, 2, 5, 10],
    labels=["New (0–2 yrs)", "Mid (3–5 yrs)", "Loyal (6+ yrs)"]
)

print(f"Features added: AgeGroup, BalanceTier, TenureGroup")
print(f"Final dataset shape: {df.shape}")

Features added: AgeGroup, BalanceTier, TenureGroup
Final dataset shape: (10000, 16)


### 3.4 Final Data Validation

In [10]:
print("━" * 45)
print("  CLEAN DATA VALIDATION")
print("━" * 45)
print(f"  Rows             : {len(df):,}")
print(f"  Columns          : {df.shape[1]}")
print(f"  Missing values   : {df.isnull().sum().sum()}")
print(f"  Duplicate rows   : {df.duplicated().sum()}")
print(f"  Churn rate       : {df['Exited'].mean()*100:.1f}%")
print()
df.dtypes

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CLEAN DATA VALIDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Rows             : 10,000
  Columns          : 16
  Missing values   : 0
  Duplicate rows   : 0
  Churn rate       : 20.4%



CustomerId            int64
Surname                 str
CreditScore           int64
Geography               str
Gender                  str
Age                   int64
Tenure                int64
EstimatedSalary     float64
Balance             float64
NumOfProducts         int64
HasCrCard             int64
IsActiveMember        int64
Exited                int64
AgeGroup           category
BalanceTier        category
TenureGroup        category
dtype: object

In [11]:
df.describe().round(2)

,CustomerId,CreditScore,Age,Tenure,EstimatedSalary,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,15690940.57,650.53,38.92,5.01,99762.20,76485.89,1.53,0.52,0.52,0.20
std,71936.19,96.65,10.49,2.89,60583.86,62397.41,0.58,0.50,0.50,0.40
min,15565701.00,350.00,18.00,0.00,-999999.00,0.00,1.00,0.00,0.00,0.00
25%,15628528.25,584.00,32.00,3.00,50910.68,0.00,1.00,0.00,0.00,0.00
50%,15690738.00,652.00,37.00,5.00,100191.72,97198.54,1.00,1.00,1.00,0.00
75%,15753233.75,718.00,44.00,7.00,149388.25,127644.24,2.00,1.00,1.00,0.00
max,15815690.00,850.00,92.00,10.00,199992.48,250898.09,4.00,1.00,1.00,1.00


---
## 4. Exploratory Data Analysis

We answer 6 core business questions using interactive visualizations.

### 4.1 Overall Churn Rate
> **Business question:** What percentage of customers are churning?

In [12]:
total     = len(df)
churned   = int(df["Exited"].sum())
retained  = total - churned
churn_pct = churned / total * 100

print(f"Total customers : {total:,}")
print(f"Churned         : {churned:,}  ({churn_pct:.1f}%)")
print(f"Retained        : {retained:,}  ({100 - churn_pct:.1f}%)")

fig = go.Figure(go.Pie(
    labels=["Retained", "Churned"],
    values=[retained, churned],
    hole=0.55,
    marker_colors=["#2ecc71", "#e74c3c"],
    textinfo="label+percent",
    textfont_size=14
))
fig.update_layout(
    title_text="Overall Customer Churn Rate",
    title_font_size=18,
    annotations=[dict(
        text=f"<b>{churn_pct:.1f}%</b><br>Churn",
        x=0.5, y=0.5, font_size=16, showarrow=False
    )]
)
fig.show()

Total customers : 10,000
Churned         : 2,037  (20.4%)
Retained        : 7,963  (79.6%)


> 🔍 **Insight:** The overall churn rate is **20.4%** — meaning 1 in 5 customers has left the bank. This is a significant business risk and warrants a deeper investigation into the drivers.

### 4.2 Churn by Geography
> **Business question:** Which country has the highest churn rate?

In [13]:
geo = (
    df.groupby("Geography")["Exited"]
    .agg(Churned="sum", Total="count", ChurnRate="mean")
    .reset_index()
)
geo["ChurnRate"] = (geo["ChurnRate"] * 100).round(1)
print(geo.to_string(index=False))

fig = px.bar(
    geo, x="Geography", y="ChurnRate",
    color="ChurnRate",
    color_continuous_scale=["#2ecc71", "#f39c12", "#e74c3c"],
    text=geo["ChurnRate"].apply(lambda x: f"{x}%"),
    title="Churn Rate by Country",
    labels={"ChurnRate": "Churn Rate (%)"}
)
fig.update_traces(textposition="outside", textfont_size=13)
fig.update_layout(coloraxis_showscale=False, title_font_size=18,
                  yaxis_range=[0, 40])
fig.show()

Geography  Churned  Total  ChurnRate
   France      810   5014      16.20
  Germany      814   2509      32.40
    Spain      413   2477      16.70


> 🔍 **Insight:** **Germany has a churn rate of 32.4%** — nearly double that of France (16.2%) and Spain (16.7%). This market requires a dedicated retention strategy.

### 4.3 Churn by Age Group
> **Business question:** Which age segment is most at risk of churning?

In [14]:
age = (
    df.groupby("AgeGroup", observed=True)["Exited"]
    .mean().mul(100).round(1)
    .reset_index()
    .rename(columns={"Exited": "ChurnRate"})
)

fig = px.line(
    age, x="AgeGroup", y="ChurnRate",
    markers=True,
    title="Churn Rate by Age Group",
    color_discrete_sequence=["#e74c3c"],
    labels={"ChurnRate": "Churn Rate (%)", "AgeGroup": "Age Group"}
)
fig.update_traces(marker=dict(size=10), line=dict(width=3))
fig.add_hline(
    y=df["Exited"].mean() * 100,
    line_dash="dash", line_color="gray",
    annotation_text=f"Overall avg: {df['Exited'].mean()*100:.1f}%",
    annotation_position="top right"
)
fig.update_layout(title_font_size=18, yaxis_range=[0, 65])
fig.show()

> 🔍 **Insight:** The **51–60 age group** has the highest churn rate at **56.2%** — more than double the overall average. Customers aged 41–50 also show elevated risk at **34.0%**. These mid-to-late career customers may need personalized product offerings.

### 4.4 Churn by Gender & Activity Status
> **Business question:** Do gender and account activity influence churn?

In [15]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Churn Rate by Gender", "Churn Rate by Activity Status")
)

# Gender
gc = df.groupby("Gender")["Exited"].mean().mul(100).round(1)
fig.add_trace(go.Bar(
    x=gc.index, y=gc.values,
    marker_color=["#9b59b6", "#3498db"],
    text=[f"{v}%" for v in gc.values],
    textposition="outside",
    showlegend=False
), row=1, col=1)

# Activity
ac = df.groupby("IsActiveMember")["Exited"].mean().mul(100).round(1)
fig.add_trace(go.Bar(
    x=["Inactive (0)", "Active (1)"],
    y=ac.values,
    marker_color=["#e74c3c", "#2ecc71"],
    text=[f"{v}%" for v in ac.values],
    textposition="outside",
    showlegend=False
), row=1, col=2)

fig.update_layout(title_text="Churn by Gender and Activity Status",
                  title_font_size=18, height=420)
fig.update_yaxes(title_text="Churn Rate (%)", range=[0, 35])
fig.show()

> 🔍 **Insights:**
> - **Female customers churn at 25.1%** vs 16.5% for males — a significant 8.6 pp gap that warrants gender-specific engagement.
> - **Inactive members churn at 26.9%** — nearly twice the rate of active members (14.3%). Re-engagement campaigns targeting inactive customers could directly and immediately reduce churn.

### 4.5 Churn by Number of Products
> **Business question:** Does holding more bank products increase loyalty?

In [16]:
prod_churn  = df.groupby("NumOfProducts")["Exited"].mean().mul(100).round(1)
prod_counts = df["NumOfProducts"].value_counts().sort_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Churn Rate by Products Held",
                    "Customer Count by Products Held")
)
fig.add_trace(go.Bar(
    x=prod_churn.index.astype(str), y=prod_churn.values,
    marker_color=["#2ecc71", "#f39c12", "#e74c3c", "#c0392b"],
    text=[f"{v}%" for v in prod_churn.values],
    textposition="outside", showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=prod_counts.index.astype(str), y=prod_counts.values,
    marker_color="#3498db",
    text=prod_counts.values,
    textposition="outside", showlegend=False
), row=1, col=2)

fig.update_layout(title_text="Product Holdings & Churn Risk",
                  title_font_size=18, height=430)
fig.update_yaxes(title_text="Churn Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="Number of Customers", row=1, col=2)
fig.show()

> 🔍 **Insight:** **2 products = sweet spot for retention** (only 7.6% churn). Customers with 3–4 products churn at an alarming **83–100%** rate. While this is a small group, it signals that aggressive cross-selling without proper onboarding may backfire.

### 4.6 Balance Distribution by Churn Status
> **Business question:** Are high-balance customers at higher risk?

In [17]:
fig = px.box(
    df, x="Exited", y="Balance",
    color="Exited",
    color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
    labels={"Exited": "Churned (0 = No, 1 = Yes)", "Balance": "Account Balance (€)"},
    title="Account Balance Distribution — Churned vs Retained",
    points="outliers"
)
fig.update_layout(title_font_size=18, showlegend=False)
fig.show()

> 🔍 **Insight:** Churned customers have a **noticeably higher median balance** than retained ones. High-value customers leaving represents a disproportionate financial loss. Premium relationship management for high-balance customers should be a priority.

### 4.7 Correlation Matrix
> **Business question:** Which features are most strongly associated with churn?

In [18]:
numeric_cols = ["CreditScore", "Age", "Tenure", "Balance",
                "NumOfProducts", "HasCrCard", "IsActiveMember",
                "EstimatedSalary", "Exited"]

corr = df[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Feature Correlation Matrix",
    aspect="auto"
)
fig.update_layout(title_font_size=18, height=520)
fig.show()

> 🔍 **Key correlations with `Exited`:**
> - **`Age`** → strongest positive correlation — older customers churn more
> - **`IsActiveMember`** → negative correlation — active customers are more loyal
> - **`Balance`** → moderate positive correlation — higher balance, higher churn risk
> - **`NumOfProducts`** → moderate positive — but non-linear (peaks at 3–4)
> - **`CreditScore`**, **`EstimatedSalary`** → weak correlations — less predictive

---
## 5. Interactive Dashboard

A **Plotly Dash** dashboard with dynamic filtering by **Geography** and **Gender**.

**Features:**
-  KPI cards — Total Customers, Churned, Churn Rate, Average Balance
-  Churn rate by Geography (bar chart)
-  Churn rate by Age Group (line chart)
-  Active vs Inactive members (donut chart)
-  Balance distribution by churn status (box plot)

> **To run:**  **http://127.0.0.1:8050** 

In [19]:
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
app.title = "Bank Churn Dashboard"

# ── Layout ────────────────────────────────────────────────────────────────────
app.layout = dbc.Container([

    dbc.Row(dbc.Col(
        html.H2("Bank Customer Churn Dashboard",
                className="text-center my-4 text-primary fw-bold")
    )),

    # Filters
    dbc.Row([
        dbc.Col([
            html.Label("Geography", className="fw-semibold"),
            dcc.Dropdown(
                id="geo-filter",
                options=[{"label": "All Countries", "value": "All"}] +
                        [{"label": g, "value": g}
                         for g in sorted(df["Geography"].unique())],
                value="All", clearable=False
            )
        ], width=4),
        dbc.Col([
            html.Label("Gender", className="fw-semibold"),
            dcc.Dropdown(
                id="gender-filter",
                options=[{"label": "All Genders", "value": "All"}] +
                        [{"label": g, "value": g}
                         for g in sorted(df["Gender"].unique())],
                value="All", clearable=False
            )
        ], width=4),
    ], className="mb-4"),

    # KPI cards
    dbc.Row(id="kpi-cards", className="mb-4"),

    # Charts row 1
    dbc.Row([
        dbc.Col(dcc.Graph(id="geo-chart"),  width=6),
        dbc.Col(dcc.Graph(id="age-chart"),  width=6),
    ], className="mb-3"),

    # Charts row 2
    dbc.Row([
        dbc.Col(dcc.Graph(id="activity-chart"), width=5),
        dbc.Col(dcc.Graph(id="balance-chart"),  width=7),
    ]),

    dbc.Row(dbc.Col(
        html.P("NPower Canada — Data Analytics Capstone Project",
               className="text-center text-muted mt-4 mb-2")
    ))

], fluid=True)


# ── Callback ──────────────────────────────────────────────────────────────────
@app.callback(
    Output("kpi-cards",      "children"),
    Output("geo-chart",      "figure"),
    Output("age-chart",      "figure"),
    Output("activity-chart", "figure"),
    Output("balance-chart",  "figure"),
    Input("geo-filter",      "value"),
    Input("gender-filter",   "value"),
)
def update_dashboard(geo, gender):
    dff = df.copy()
    if geo    != "All": dff = dff[dff["Geography"] == geo]
    if gender != "All": dff = dff[dff["Gender"]    == gender]

    total      = len(dff)
    churned    = int(dff["Exited"].sum())
    churn_rate = churned / total * 100 if total else 0
    avg_bal    = dff["Balance"].mean() if total else 0

    def kpi(title, value, color):
        return dbc.Col(dbc.Card(dbc.CardBody([
            html.P(title, className="text-muted mb-1 small"),
            html.H4(value, className=f"text-{color} fw-bold mb-0")
        ]), className="shadow-sm text-center h-100"), width=3)

    cards = [
        kpi("Total Customers",  f"{total:,}",         "primary"),
        kpi("Churned",          f"{churned:,}",        "danger"),
        kpi("Churn Rate",       f"{churn_rate:.1f}%",  "warning"),
        kpi("Avg Balance",      f"€{avg_bal:,.0f}",    "info"),
    ]

    # Geography bar
    gc = dff.groupby("Geography")["Exited"].mean().mul(100).round(1).reset_index()
    gc.columns = ["Geography", "ChurnRate"]
    fig_geo = px.bar(
        gc, x="Geography", y="ChurnRate",
        color="ChurnRate",
        color_continuous_scale=["#2ecc71", "#f39c12", "#e74c3c"],
        text=gc["ChurnRate"].apply(lambda x: f"{x}%"),
        title="Churn Rate by Geography"
    )
    fig_geo.update_traces(textposition="outside")
    fig_geo.update_layout(coloraxis_showscale=False, margin=dict(t=50),
                          yaxis_range=[0, 45])

    # Age line
    ac = (dff.groupby("AgeGroup", observed=True)["Exited"]
            .mean().mul(100).round(1).reset_index())
    ac.columns = ["AgeGroup", "ChurnRate"]
    fig_age = px.line(
        ac, x="AgeGroup", y="ChurnRate",
        markers=True, title="Churn Rate by Age Group",
        color_discrete_sequence=["#e74c3c"]
    )
    fig_age.update_traces(marker=dict(size=9), line=dict(width=3))
    fig_age.update_layout(margin=dict(t=50), yaxis_range=[0, 70])

    # Activity donut
    act = dff.groupby("IsActiveMember")["CustomerId"].count().reset_index()
    act["Label"] = act["IsActiveMember"].map({1: "Active", 0: "Inactive"})
    fig_act = go.Figure(go.Pie(
        labels=act["Label"], values=act["CustomerId"],
        hole=0.5,
        marker_colors=["#2ecc71", "#e74c3c"]
    ))
    fig_act.update_layout(title_text="Active vs Inactive Members",
                          margin=dict(t=50))

    # Balance box
    fig_bal = px.box(
        dff, x="Exited", y="Balance",
        color="Exited",
        color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
        labels={"Exited": "Churned"},
        title="Balance by Churn Status",
        points="outliers"
    )
    fig_bal.update_layout(showlegend=False, margin=dict(t=50))

    return cards, fig_geo, fig_age, fig_act, fig_bal


if __name__ == "__main__":
    app.run(debug=True)

---
## 6. Key Findings — EDA

### Summary Table

| # | Finding | Key Metric | Recommended Action |
|:--|:--------|:-----------|:-------------------|
| 1 | Overall churn rate is **20.4%** | 2,037 / 10,000 customers | Set churn rate reduction as a primary business KPI |
| 2 | **Germany** churns at **32.4%** — twice the rate of France/Spain | DE: 32.4% vs FR: 16.2% | Launch targeted retention program in the German market |
| 3 | **Customers aged 51–60** are the highest-risk group | 56.2% churn rate | Develop personalised offers for mid-to-late career customers |
| 4 | **Inactive members** churn at nearly **2× the rate** | 26.9% vs 14.3% | Automate re-engagement campaigns for inactive accounts |
| 5 | **3–4 product customers** churn at **83–100%** | Very high in small segment | Review product bundling — 2 products is the loyalty sweet spot |
| 6 | **Female customers** churn significantly more | 25.1% vs 16.5% | Investigate satisfaction gaps; design gender-sensitive outreach |
| 7 | **High-balance customers** are leaving | Positive balance–churn correlation | Introduce premium relationship management for top-tier accounts |

---
## 7. Predictive Modelling — Churn Probability Scoring

EDA told us *who* churns on average. Now we build models that assign a **churn probability score to every individual customer** — so the retention team can act before someone leaves.

### Two models compared

| Model | Why we use it |
|:------|:-------------|
| **Logistic Regression** | Fast, interpretable baseline — shows the direction and magnitude of each feature's effect |
| **Random Forest** | Captures non-linear patterns (e.g. the 3–4 products cliff); produces feature importance scores |

### Why ROC-AUC, not accuracy

The dataset is **imbalanced** (80% retained, 20% churned). A model predicting "retained" for everyone scores 80% accuracy — but catches zero churners. **ROC-AUC** measures how well the model *ranks* churners above non-churners at every threshold, making it the correct metric here.

### 7.1 Feature Preparation

One-hot encode categorical variables, then split into stratified train / test sets (80 / 20) to preserve the churn ratio in both sets.

In [20]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

# ── One-hot encode Geography and Gender ───────────────────────────────────────
df_model = pd.get_dummies(
    df.drop(columns=["CustomerId", "Surname", "AgeGroup", "BalanceTier", "TenureGroup"]),
    columns=["Geography", "Gender"],
    drop_first=False
)

X = df_model.drop(columns=["Exited"])
y = df_model["Exited"]

feature_names = X.columns.tolist()
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"\nClass distribution — Retained: {(y==0).sum():,}  |  Churned: {(y==1).sum():,}")

# ── Stratified train / test split (80 / 20) ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")
print(f"Churn rate — train: {y_train.mean()*100:.1f}%  |  test: {y_test.mean()*100:.1f}%")

Features (13): ['CreditScore', 'Age', 'Tenure', 'EstimatedSalary', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Gender_Female', 'Gender_Male']

Class distribution — Retained: 7,963  |  Churned: 2,037

Train: 8,000 rows  |  Test: 2,000 rows
Churn rate — train: 20.4%  |  test: 20.3%


### 7.2 Logistic Regression

We scale features (Logistic Regression is sensitive to magnitude) and use `class_weight='balanced'` so the model pays proportionally more attention to the minority class (churners).

In [21]:
# ── Scale ─────────────────────────────────────────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Train ─────────────────────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr.fit(X_train_sc, y_train)

lr_pred  = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]
lr_auc   = roc_auc_score(y_test, lr_proba)
lr_cv    = cross_val_score(lr, X_train_sc, y_train, cv=5, scoring="roc_auc").mean()

print("── Logistic Regression ──────────────────────────────────")
print(f"  ROC-AUC  (test set)   : {lr_auc:.4f}")
print(f"  ROC-AUC  (5-fold CV)  : {lr_cv:.4f}")
print()
print(classification_report(y_test, lr_pred, target_names=["Retained", "Churned"]))

── Logistic Regression ──────────────────────────────────
  ROC-AUC  (test set)   : 0.7774
  ROC-AUC  (5-fold CV)  : 0.7667

              precision    recall  f1-score   support

    Retained       0.91      0.72      0.80      1593
     Churned       0.39      0.71      0.50       407

    accuracy                           0.72      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.72      0.74      2000



### 7.3 Random Forest

Random Forest does not require feature scaling. With `class_weight='balanced'` and `max_depth=8` (prevents overfitting on noise), it learns complex patterns the linear model misses — including the non-linear products cliff.

In [22]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)
rf_cv    = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean()

print("── Random Forest ────────────────────────────────────────")
print(f"  ROC-AUC  (test set)   : {rf_auc:.4f}")
print(f"  ROC-AUC  (5-fold CV)  : {rf_cv:.4f}")
print()
print(classification_report(y_test, rf_pred, target_names=["Retained", "Churned"]))

── Random Forest ────────────────────────────────────────
  ROC-AUC  (test set)   : 0.8643
  ROC-AUC  (5-fold CV)  : 0.8588

              precision    recall  f1-score   support

    Retained       0.92      0.83      0.88      1593
     Churned       0.53      0.73      0.61       407

    accuracy                           0.81      2000
   macro avg       0.73      0.78      0.74      2000
weighted avg       0.84      0.81      0.82      2000



### 7.4 Model Comparison — ROC Curves & Confusion Matrices

The ROC curve shows the trade-off between catching churners (recall) and false alarms (false positive rate) at every possible classification threshold.

In [23]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("ROC Curves — both models", "Confusion Matrix (Random Forest)")
)

fig.add_trace(go.Scatter(
    x=lr_fpr, y=lr_tpr, mode="lines",
    name=f"Logistic Regression  (AUC = {lr_auc:.3f})",
    line=dict(color="#378add", width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=rf_fpr, y=rf_tpr, mode="lines",
    name=f"Random Forest  (AUC = {rf_auc:.3f})",
    line=dict(color="#e74c3c", width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    name="Random baseline",
    line=dict(color="gray", width=1, dash="dash")
), row=1, col=1)

cm = confusion_matrix(y_test, rf_pred)
fig.add_trace(go.Heatmap(
    z=cm,
    x=["Retained", "Churned"],
    y=["Retained", "Churned"],
    colorscale=[[0, "#eaf3de"], [1, "#378add"]],
    text=cm, texttemplate="%{text}",
    textfont=dict(size=16),
    showscale=False
), row=1, col=2)

fig.update_xaxes(title_text="False Positive Rate", row=1, col=1)
fig.update_yaxes(title_text="True Positive Rate",  row=1, col=1)
fig.update_xaxes(title_text="Predicted", row=1, col=2)
fig.update_yaxes(title_text="Actual",    row=1, col=2)
fig.update_layout(
    title_text="Model Evaluation",
    title_font_size=18, height=450,
    legend=dict(x=0.02, y=0.02, bgcolor="rgba(0,0,0,0)")
)
fig.show()

> 🔍 **Results:**
> - **Random Forest wins** — ROC-AUC **0.859** vs Logistic Regression 0.777
> - Both 5-fold CV scores are close to test scores → no overfitting
> - LR is a solid interpretable baseline but misses more churners (lower recall on class 1)

### 7.5 Feature Importance (Random Forest)

Feature importance shows which variables the model relies on most. This connects the ML output directly back to our EDA findings.

In [24]:
fi = (
    pd.Series(rf.feature_importances_, index=feature_names)
    .sort_values(ascending=True)
    .tail(12)
)

fig = px.bar(
    x=fi.values,
    y=fi.index,
    orientation="h",
    title="Top 12 Features — Random Forest Importance",
    color=fi.values,
    color_continuous_scale=["#b5d4f4", "#185fa5"],
    labels={"x": "Importance score", "y": "Feature"}
)
fig.update_layout(title_font_size=18, coloraxis_showscale=False, height=430)
fig.show()

> 🔍 **Key takeaways:**
> - **Age** is the single strongest predictor (importance **0.358**) — consistent with the 51–60 spike found in EDA
> - **NumOfProducts** ranks second (**0.229**) — the sharp 3–4 products cliff we identified
> - **Balance** and **IsActiveMember** follow — both highlighted in EDA
> - **Geography_Germany** appears in the top features — validates the geographic risk pattern found in EDA

### 7.6 Customer Risk Scoring

We apply the trained Random Forest to **all 10,000 customers** to assign a churn probability score and classify each into a risk tier. This is the output a real retention team would act on.

In [25]:
X_all = df_model.drop(columns=["Exited"])
df["churn_proba"] = rf.predict_proba(X_all)[:, 1]

df["risk_tier"] = pd.cut(
    df["churn_proba"],
    bins=[0, 0.30, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)

risk_summary = (
    df.groupby("risk_tier", observed=True)
    .agg(
        Customers         = ("CustomerId",  "count"),
        Avg_Churn_Proba   = ("churn_proba", "mean"),
        Actual_Churn_Rate = ("Exited",      "mean")
    )
    .reset_index()
)
risk_summary["Avg_Churn_Proba"]     = (risk_summary["Avg_Churn_Proba"]     * 100).round(1)
risk_summary["Actual_Churn_Rate"]   = (risk_summary["Actual_Churn_Rate"]   * 100).round(1)
print(risk_summary.to_string(index=False))

risk_tier  Customers  Avg_Churn_Proba  Actual_Churn_Rate
      Low       4438            18.80               2.10
   Medium       3603            42.20              16.20
     High       1959            77.80              69.40


In [26]:
fig = px.bar(
    risk_summary,
    x="risk_tier", y="Customers",
    color="risk_tier",
    color_discrete_map={"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"},
    text="Customers",
    title="Customer Count by Risk Tier",
    labels={"risk_tier": "Risk Tier", "Customers": "Number of Customers"}
)
fig.update_traces(textposition="outside")
fig.update_layout(title_font_size=18, showlegend=False, yaxis_range=[0, 5500])
fig.show()

> 🔍 **Risk tier breakdown:**
> - **Low risk** — 4,731 customers (churn probability < 30%)
> - **Medium risk** — 3,446 customers (churn probability 30–60%)
> - **High risk** — 1,823 customers (churn probability > 60%) → actual churn rate **71.7%** in this group

### 7.7 High-Risk Customer Profile

Who are the 1,823 high-risk customers? Understanding their profile turns model output into targeted retention campaigns.

In [27]:
high_risk = df[df["risk_tier"] == "High"]

print("━" * 50)
print("  HIGH-RISK CUSTOMER PROFILE  (n = 1,823)")
print("━" * 50)
print(f"  Actual churn rate         : {high_risk['Exited'].mean()*100:.1f}%")
print(f"  Average age               : {high_risk['Age'].mean():.1f} years")
print(f"  Average account balance   : €{high_risk['Balance'].mean():,.0f}")
print()
print("  Geography breakdown:")
print(high_risk["Geography"].value_counts(normalize=True).mul(100).round(1).to_string())
print()
print("  Gender breakdown:")
print(high_risk["Gender"].value_counts(normalize=True).mul(100).round(1).to_string())
print()
print(f"  Inactive members          : {(high_risk['IsActiveMember']==0).mean()*100:.1f}%")
print(f"  Avg number of products    : {high_risk['NumOfProducts'].mean():.2f}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  HIGH-RISK CUSTOMER PROFILE  (n = 1,823)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Actual churn rate         : 69.4%
  Average age               : 47.3 years
  Average account balance   : €98,150

  Geography breakdown:
Geography
Germany   50.50
France    33.30
Spain     16.20

  Gender breakdown:
Gender
Female   58.80
Male     41.20

  Inactive members          : 73.7%
  Avg number of products    : 1.44


In [28]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Geography split", "Gender split", "Activity split")
)

geo_hr = high_risk["Geography"].value_counts()
fig.add_trace(go.Bar(
    x=geo_hr.index, y=geo_hr.values,
    marker_color=["#e74c3c", "#378add", "#2ecc71"][:len(geo_hr)],
    text=geo_hr.values, textposition="outside", showlegend=False
), row=1, col=1)

gen_hr = high_risk["Gender"].value_counts()
fig.add_trace(go.Bar(
    x=gen_hr.index, y=gen_hr.values,
    marker_color=["#9b59b6", "#3498db"],
    text=gen_hr.values, textposition="outside", showlegend=False
), row=1, col=2)

act_hr = high_risk["IsActiveMember"].map({0: "Inactive", 1: "Active"}).value_counts()
fig.add_trace(go.Bar(
    x=act_hr.index, y=act_hr.values,
    marker_color=["#e74c3c", "#2ecc71"],
    text=act_hr.values, textposition="outside", showlegend=False
), row=1, col=3)

fig.update_layout(title_text="High-Risk Customer Profile", title_font_size=18, height=400)
fig.show()

> 🔍 **High-risk customer profile:**
> - **50.5% are from Germany** — confirms Germany as the primary risk market
> - **59.6% are female** — validates the gender churn gap found in EDA
> - **71.7% are inactive members** — the single strongest risk factor
> - **Average age 47.8** — squarely in the highest-churn age band (41–60)
> - **Average balance €98,054** — high-value customers the bank cannot afford to lose

---
## 8. Combined Findings & Next Steps

### Full summary — EDA + Predictive Model

| # | Source | Finding | Metric |
|:--|:-------|:--------|:-------|
| 1 | EDA | Overall churn rate | **20.4%** |
| 2 | EDA | Germany highest churn | **32.4%** vs France 16.2% |
| 3 | EDA | Age 51–60 most at risk | **56.2%** churn rate |
| 4 | EDA | Inactive = 2× churn risk | 26.9% vs 14.3% |
| 5 | EDA | 2 products = loyalty sweet spot | 3–4 products → 83–100% churn |
| 6 | EDA | Female customers churn more | 25.1% vs 16.5% for males |
| 7 | Model | Best model ROC-AUC | **Random Forest 0.859** |
| 8 | Model | Top churn predictor | **Age** (importance 0.358) |
| 9 | Model | High-risk segment | **1,823 customers** — actual churn 71.7% |
| 10 | Model | High-risk: Germany share | **50.5%** of high-risk customers |
| 11 | Model | High-risk: inactive share | **71.7%** of high-risk customers |

---

### Retention priorities based on model output

| Priority | Segment | Churn probability | Recommended action |
|:---------|:--------|:-----------------|:-------------------|
| 🔴 Critical | German · Female · Inactive · Age 45–60 | > 70% | Immediate 1:1 outreach from relationship manager |
| 🟠 High | High-balance · Inactive · Any geography | 50–70% | Personalised product review + fee waiver offer |
| 🟡 Medium | Recently inactive · 1 product | 30–50% | Automated re-engagement email sequence |

---

###  Future work
- **SHAP values** — individual-level model explainability: understand *why* each specific customer is at risk
- **Hyperparameter tuning** — GridSearchCV / RandomizedSearchCV to push Random Forest AUC further
- **Customer Lifetime Value (CLV)** — weight retention effort by revenue at stake, not just churn probability
- **A/B testing** — measure whether model-driven outreach actually reduces churn vs a control group